# Segmentación de Grietas en Video (PIDNet & YOLO-Seg)

Este notebook permite ejecutar la inferencia de video de forma unificada en Google Colab para comparar **PIDNet** y **YOLOv11-Seg** sobre un mismo hardware (GPU T4/V100/A100).

### Características:
- **Preservación de Aspect Ratio**: No deforma videos 16:9 a cuadrados.
- **PIDNet**: Resolución dinámica con múltiplo de 32 y normalización ImageNet.
- **YOLO-Seg**: Máscaras de alta resolución alineadas con `retina_masks=True`.
- **Métricas de Rendimiento**: Mide frames totales, tiempo total, FPS de pipeline y FPS neto de inferencia.

## 1. Montar Google Drive y ubicarse en el proyecto

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Ajusta esta ruta a la ubicación de tu proyecto en Drive
project_dir = '/content/drive/MyDrive/tp_computer_vision_ii'
if os.path.exists(project_dir):
    os.chdir(project_dir)
    print("Directorio actual:", os.getcwd())
else:
    print(f"ADVERTENCIA: No se encontró {project_dir}. Verifica la ruta.")

## 2. Instalar dependencias necesarias (si no están en el entorno)

In [ ]:
!pip install -q ultralytics opencv-python pyyaml

## 3. Ejecutar Inferencia de Video

Puedes ejecutar la inferencia llamando al script unificado `src/unificado/infer_video.py`.

### Opción A: Inferencia con PIDNet

In [ ]:
# Inferencia PIDNet
# Para videos 16:9 (ej. 768x432 o 1080p o 4K), target-height=448 procesa a 768x448 (múltiplos de 32)
!python src/unificado/infer_video.py \
    --config configs/pidnet.yaml \
    --weights src/pidnet/best_pidnet_S.pth \
    --input datasets/videos/video_cracks_2.mp4 \
    --output /content/drive/MyDrive/salidas/video_pidnet.mp4 \
    --target-height 448 \
    --alpha 0.45

### Opción B: Inferencia con YOLOv11-Seg

In [ ]:
# Inferencia YOLO-Seg
!python src/unificado/infer_video.py \
    --config configs/yolo_seg.yaml \
    --weights runs/yolo_seg/yolo11n_seg_nano/weights/best.pt \
    --input datasets/videos/video_cracks_2.mp4 \
    --output /content/drive/MyDrive/salidas/video_yolo.mp4 \
    --conf 0.10 \
    --imgsz 640 \
    --alpha 0.45

## 4. Visualizar un frame de comparación directamente en Colab

In [ ]:
import cv2
import matplotlib.pyplot as plt

def show_video_frame(video_path, frame_idx=50):
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    cap.release()
    if ret:
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(10, 6))
        plt.imshow(frame_rgb)
        plt.title(f"{os.path.basename(video_path)} - Frame {frame_idx}")
        plt.axis('off')
        plt.show()
    else:
        print(f"No se pudo leer el frame {frame_idx} de {video_path}")

# Ejemplo de visualización:
# show_video_frame('/content/drive/MyDrive/salidas/video_pidnet.mp4', frame_idx=100)
# show_video_frame('/content/drive/MyDrive/salidas/video_yolo.mp4', frame_idx=100)